# micrograd — a tiny scalar autograd engine

Following Karpathy's *The spelled-out intro to neural networks and backpropagation*.
A `Value` wraps one number and records the operation that produced it, so the whole
computation forms a graph we can differentiate by walking backwards through it.

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from graphviz import Digraph
%matplotlib inline

## The autograd engine

Each operator (`+`, `*`, `**`, `tanh`, `exp`) does two things: computes the forward
value, and stores a `_backward` closure that knows how to push this node's gradient
onto its inputs (the *local* derivative times the incoming grad). `backward()` seeds
`dL/dL = 1` and calls those closures in reverse topological order.

In [ ]:
class Value:
    """A scalar node in a computation graph, with reverse-mode autograd."""

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0                # dL/d(self); filled in by backward()
        self._backward = lambda: None  # closure: push this node's grad to its inputs
        self._prev = set(_children)    # the Values that produced this one
        self._op = _op                 # the op that produced this one (for the viz)
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # '+' distributes the incoming gradient unchanged to both inputs
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # local derivative of a product is the *other* operand's value
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers supported"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            # d/dx tanh(x) = 1 - tanh(x)^2, reusing the forward output t
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad   # d/dx e^x = e^x
        out._backward = _backward
        return out

    # --- niceties so ordinary arithmetic (with plain numbers too) just works ---
    def __neg__(self):            return self * -1
    def __radd__(self, other):    return self + other
    def __sub__(self, other):     return self + (-other)
    def __rsub__(self, other):    return other + (-self)
    def __rmul__(self, other):    return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other):return other * self**-1

    def backward(self):
        # 1) topological order: every node appears after all of its inputs
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        # 2) seed dL/dL = 1, then apply the chain rule in reverse
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

## Visualizing the graph

In [ ]:
def trace(root):
    # collect every node and edge reachable from root
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # left to right
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

## A single neuron

`o = tanh(x1*w1 + x2*w2 + b)`. Call `o.backward()` and every node's `.grad` fills in —
that's `do/d(node)` for the whole graph in one backward sweep.

In [ ]:
x1 = Value(2.0, label='x1'); x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1'); w2 = Value(1.0, label='w2')
b  = Value(6.8813735870195432, label='b')

n = x1*w1 + x2*w2 + b; n.label = 'n'
o = n.tanh();          o.label = 'o'

o.backward()
draw_dot(o)

## Building an MLP

A `Neuron` is weights + bias + a tanh. A `Layer` is a list of neurons. An `MLP` is a
list of layers. `parameters()` gathers every `Value` we want to train.

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)  # w . x + b
        return act.tanh()
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Training

Gradient descent: forward pass -> mean-squared-error loss -> **zero the grads** ->
`backward()` -> nudge every parameter downhill. Zeroing matters because grads
*accumulate* (`+=`); forget it and each step adds to the last step's gradient.

In [ ]:
model = MLP(3, [4, 4, 1])   # 3 inputs -> two hidden layers of 4 -> 1 output

xs = [[2.0,  3.0, -1.0],
      [3.0, -1.0,  0.5],
      [0.5,  1.0,  1.0],
      [1.0,  1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]   # desired targets

for step in range(100):
    # forward: predictions + loss
    ypred = [model(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward: zero grads FIRST (they accumulate), then backprop
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()

    # update: step every parameter downhill
    for p in model.parameters():
        p.data += -0.05 * p.grad

    if step % 10 == 0:
        print(step, loss.data)

print('predictions:', [round(y.data, 3) for y in ypred])